# Hierarchical Clustering on Well Log Data
## Machine learning for energy applications — Module 3, Exercise 4

In this exercise, we will apply **Hierarchical (Agglomerative) Clustering** to well log data consisting of two measurements: **DTC** (compressional sonic slowness) and **RHOB** (bulk density). These two logs are commonly used in petrophysics to distinguish lithologies and fluid types.

Hierarchical clustering builds a tree of clusters (a **dendrogram**) by iteratively merging the closest clusters. Unlike K-Means, it does not require specifying the number of clusters upfront — instead, we can visually inspect the dendrogram and choose a cut height.

**Topics covered:**
1. Load and explore the dataset
2. Standardize features
3. Compute the linkage matrix (Ward method)
4. Plot and interpret the dendrogram
5. Apply AgglomerativeClustering
6. Visualize clusters
7. Compare linkage methods
8. Compare with K-Means
9. Summary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['figure.dpi'] = 100

---
## 1. Load and Explore the Dataset

Load `hierarchical_clustering_dataset.csv` and inspect its shape, statistics, and create a scatter plot of DTC vs RHOB.

**Tasks:**
- Load the CSV into a DataFrame
- Print shape, column names, missing values, and summary statistics
- Display the first few rows

In [ ]:
# YOUR CODE HERE
# Hint: df = pd.read_csv('hierarchical_clustering_dataset.csv')
# Print df.shape, df.columns, df.isnull().sum(), df.describe()


**Tasks:**
- Create a scatter plot of DTC (x-axis) vs RHOB (y-axis)
- Create histograms for each feature

In [ ]:
# YOUR CODE HERE
# Hint: plt.scatter(df['DTC'], df['RHOB'], alpha=0.5, s=20)


In [ ]:
# YOUR CODE HERE
# Hint: fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].hist(df['DTC'], ...)
# axes[1].hist(df['RHOB'], ...)


---
## 2. Standardize Features

Since DTC and RHOB have very different scales, we standardize before clustering.

**Exploratory analysis of one fixed dataset.** The scaler is fitted on all 82 samples because we only describe these data; nothing is predicted for new samples, so there is no training/test split.

**Tasks:**
- Extract the feature matrix from `df[['DTC', 'RHOB']]`
- Fit a `StandardScaler` and transform
- Verify mean is ~0 and std is ~1

In [ ]:
# YOUR CODE HERE
# Hint: scaler = StandardScaler()
# X = df[['DTC', 'RHOB']].values
# X_scaled = scaler.fit_transform(X)


---
## 3. Compute the Linkage Matrix (Ward Method)

The linkage matrix encodes the hierarchical merging process. We use the **Ward** method, which minimizes the total within-cluster variance at each merge step.

**Tasks:**
- Use `linkage(X_scaled, method='ward')` to compute the linkage matrix
- Print its shape and first few rows

In [ ]:
# YOUR CODE HERE
# Hint: Z = linkage(X_scaled, method='ward')
# Each row: [cluster_i, cluster_j, distance, n_samples_in_new_cluster]


---
## 4. Plot the Dendrogram

The dendrogram visualizes the hierarchical merging. We can choose a **cut height** to determine the number of clusters. Choose the cut inside a large gap between successive merge heights: a long vertical branch means that groups stayed separate over a wide range of heights. Leaf order is not a physical coordinate.

**Tasks:**
- Plot the dendrogram using `dendrogram(Z, truncate_mode='lastp', p=30, ...)`
- Print the last merge heights (`Z[-10:, 2]`) and the gaps between them; choose the cut height inside the largest gap
- Add a horizontal line at the cut height
- Use `fcluster(Z, t=cut_height, criterion='distance')` to determine the resulting number of clusters

In [ ]:
# YOUR CODE HERE
# Hint: plt.figure(figsize=(14, 7))
# dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90, show_contracted=True)


In [ ]:
# YOUR CODE HERE
# Hint: last_heights = Z[-10:, 2]; print(last_heights, np.diff(last_heights))
# cut_height = ...  # inside the largest gap between successive merge heights
# plt.axhline(y=cut_height, color='red', linestyle='--')
# labels_cut = fcluster(Z, t=cut_height, criterion='distance')
# n_clusters_from_cut = len(np.unique(labels_cut))


---
## 5. Apply AgglomerativeClustering

Use scikit-learn's `AgglomerativeClustering` with the chosen number of clusters.

**Tasks:**
- Create `AgglomerativeClustering(n_clusters=..., linkage='ward')`
- Fit and predict on `X_scaled`
- Print cluster sizes and silhouette score

In [ ]:
# YOUR CODE HERE
# Hint: agg = AgglomerativeClustering(n_clusters=n_clusters_from_cut, linkage='ward')
# labels_agg = agg.fit_predict(X_scaled)
# sil = silhouette_score(X_scaled, labels_agg)


---
## 6. Visualize Clusters

Plot the clustering results in both the original feature space and the standardized space.

**Tasks:**
- Create a 1x2 subplot grid
- Left: scatter plot in original space (DTC vs RHOB), coloured by cluster
- Right: scatter plot in standardized space, coloured by cluster

In [ ]:
# YOUR CODE HERE
# Hint: fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# axes[0].scatter(df['DTC'], df['RHOB'], c=labels_agg, ...)
# axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_agg, ...)


---
## 7. Compare Linkage Methods

Compare the four main linkage methods: **Ward**, **Complete**, **Average**, and **Single**.

**Tasks:**
- Loop over `['ward', 'complete', 'average', 'single']`
- For each, fit AgglomerativeClustering and compute silhouette score
- Create a 2x2 subplot grid with scatter plots for each method
- Also create a 2x2 grid of dendrograms for each method

In [ ]:
# YOUR CODE HERE
# Hint:
# linkage_methods = ['ward', 'complete', 'average', 'single']
# fig, axes = plt.subplots(2, 2, figsize=(14, 12))
# for ax, method in zip(axes.ravel(), linkage_methods):
#     agg_m = AgglomerativeClustering(n_clusters=n_clusters, linkage=method)
#     labels_m = agg_m.fit_predict(X_scaled)
#     sil_m = silhouette_score(X_scaled, labels_m)
#     ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_m, ...)


In [ ]:
# YOUR CODE HERE
# Hint: fig, axes = plt.subplots(2, 2, figsize=(16, 12))
# for ax, method in zip(axes.ravel(), linkage_methods):
#     Z_m = linkage(X_scaled, method=method)
#     dendrogram(Z_m, truncate_mode='lastp', p=20, ax=ax, ...)


---
## 8. Compare with K-Means

Apply K-Means with the same number of clusters and compare the results side-by-side with hierarchical clustering.

**Tasks:**
- Fit `KMeans(n_clusters=..., random_state=42, n_init=10)`
- Compute silhouette scores for both methods
- Create a 1x2 subplot comparing hierarchical (left) and K-Means (right)
- Plot K-Means centroids on the right panel

In [ ]:
# YOUR CODE HERE
# Hint: km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
# labels_km = km.fit_predict(X_scaled)
# sil_km = silhouette_score(X_scaled, labels_km)


In [ ]:
# YOUR CODE HERE
# Hint: fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# Left: scatter with labels_agg
# Right: scatter with labels_km + km.cluster_centers_ as red X markers


---
## 9. Summary

In this exercise we:

1. **Loaded and explored** a well log dataset with DTC (sonic slowness) and RHOB (bulk density) measurements.
2. **Standardized** the features to ensure equal weighting in distance calculations.
3. **Computed the Ward linkage matrix** and plotted a **dendrogram** to visualize the hierarchical merging process.
4. **Chose a cut height** in the largest gap between merge heights to determine the number of clusters.
5. **Applied AgglomerativeClustering** and visualized the resulting clusters in both original and standardized feature spaces.
6. **Compared four linkage methods** (Ward, Complete, Average, Single) using silhouette scores and visual inspection.
7. **Compared hierarchical clustering with K-Means** on the same data and number of clusters.

**Key takeaways:**
- Hierarchical clustering does not require specifying K upfront — cut the dendrogram inside a large gap between successive merge heights.
- The **Ward** method tends to produce compact, evenly-sized clusters (similar to K-Means).
- **Single** linkage can produce elongated "chain" clusters, while **Complete** linkage tends to produce more compact clusters.
- For well log data, both hierarchical clustering and K-Means propose groupings of the log responses; which lithologies or fluids they correspond to, if any, is a hypothesis to test against core, cuttings or other logs.